In [3]:
import pandas as pd

# 1. Загрузка файла `/content/Enter_korea_by_age.csv`
df = pd.read_csv('/content/Enter_korea_by_age.csv')

# 2. Определение функции `fix_date`
def fix_date(x):
    try:
        y, m = str(x).split('-')
        return pd.Timestamp(year=int(y), month=int(m), day=1)
    except:
        return pd.to_datetime(x, errors='coerce')

# 3. Применение функции `fix_date` к столбцу `date`
df['date'] = df['date'].apply(fix_date)

# 4. Удаление строк с пропущенными значениями в 'date' или 'visitor'
df = df.dropna(subset=['date', 'visitor'])

# 5. Сортировка DataFrame и сброс индекса
df = df.sort_values(['nation', 'date']).reset_index(drop=True)

print("DataFrame loaded, date column processed, missing values handled, and sorted.")
print(df.head())

DataFrame loaded, date column processed, missing values handled, and sorted.
        date nation  visitor     growth     share  age0-20  age21-30  \
0 2019-01-01   *GCC     1776  -1.606648  0.160753      247       477   
1 2019-02-01   *GCC     1478  16.103692  0.122982      162       432   
2 2019-03-01   *GCC     3021   1.375839  0.196726      633       775   
3 2019-04-01   *GCC     3265  32.346980  0.199686      516       864   
4 2019-05-01   *GCC      930 -27.795031  0.062597      126       287   

   age31-40  age41-50  age51-60  age61  
0       521       289       140     70  
1       456       198       134     64  
2       773       405       278    120  
3       914       470       334    137  
4       243       144        69     38  


In [4]:
import pandas as pd

# Ensure df is available from previous step, assuming it's in the kernel state
# df = pd.read_csv('/content/Enter_korea_by_age.csv') # Don't re-read if already loaded

# === 3. Создаем target ===
# Среднее по стране
country_mean = df.groupby('nation')['visitor'].mean().rename('mean_country')
df = df.merge(country_mean, on='nation', how='left')

# Следующий месяц
df['visitor_next'] = df.groupby('nation')['visitor'].shift(-1)

# is_high_next = 1 если следующий месяц > среднего
df = df.dropna(subset=['visitor_next']).reset_index(drop=True)
df['is_high_next'] = (df['visitor_next'] > df['mean_country']).astype(int)

print("Target variable 'is_high_next' and 'visitor_next' created.")
print(df.head())

Target variable 'is_high_next' and 'visitor_next' created.
        date nation  visitor     growth     share  age0-20  age21-30  \
0 2019-01-01   *GCC     1776  -1.606648  0.160753      247       477   
1 2019-02-01   *GCC     1478  16.103692  0.122982      162       432   
2 2019-03-01   *GCC     3021   1.375839  0.196726      633       775   
3 2019-04-01   *GCC     3265  32.346980  0.199686      516       864   
4 2019-05-01   *GCC      930 -27.795031  0.062597      126       287   

   age31-40  age41-50  age51-60  age61  mean_country  visitor_next  \
0       521       289       140     70     2381.4375        1478.0   
1       456       198       134     64     2381.4375        3021.0   
2       773       405       278    120     2381.4375        3265.0   
3       914       470       334    137     2381.4375         930.0   
4       243       144        69     38     2381.4375        4736.0   

   is_high_next  
0             0  
1             1  
2             1  
3             0

In [5]:
import pandas as pd

# === 4. Создаём признаки ===
df['month'] = df['date'].dt.month

# Один предыдущий месяц
df['visitor_prev1'] = df.groupby('nation')['visitor'].shift(1)
df['visitor_prev1'] = df['visitor_prev1'].fillna(df['visitor'].median())

# OHE для nation
nation_ohe = pd.get_dummies(df['nation'], prefix='nat')

X = pd.concat([
    df[['month', 'growth', 'share', 'visitor_prev1']],
    nation_ohe
], axis=1).fillna(0)

y = df['is_high_next']

print("Features (month, growth, share, visitor_prev1, nation_ohe) and target (y) created.")
print(X.head())
print(y.head())

Features (month, growth, share, visitor_prev1, nation_ohe) and target (y) created.
   month     growth     share  visitor_prev1  nat_*GCC  nat_Africa others  \
0      1  -1.606648  0.160753         2765.0      True              False   
1      2  16.103692  0.122982         1776.0      True              False   
2      3   1.375839  0.196726         1478.0      True              False   
3      4  32.346980  0.199686         3021.0      True              False   
4      5 -27.795031  0.062597         3265.0      True              False   

   nat_America others  nat_Asia others  nat_Austrailia  nat_Austria  ...  \
0               False            False           False        False  ...   
1               False            False           False        False  ...   
2               False            False           False        False  ...   
3               False            False           False        False  ...   
4               False            False           False        False  ...  

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# === 5. Train/test split ===
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)

# Масштабирование
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Data split into training and test sets and scaled.")
print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)


Data split into training and test sets and scaled.
Shape of X_train: (720, 64)
Shape of X_test: (180, 64)


In [7]:
import tensorflow as tf

# === 6. Простая нейросеть ===
model = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu', input_shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid') # Use sigmoid for binary classification
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy']) # Using binary_crossentropy for binary classification

model.fit(X_train, y_train, epochs=20, batch_size=16, validation_split=0.2, verbose=1)

print("Neural network model built and trained.")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.5340 - loss: 0.7447 - val_accuracy: 0.5069 - val_loss: 0.7245
Epoch 2/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.5594 - loss: 0.6744 - val_accuracy: 0.6042 - val_loss: 0.6929
Epoch 3/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6761 - loss: 0.6210 - val_accuracy: 0.6181 - val_loss: 0.6702
Epoch 4/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6928 - loss: 0.6030 - val_accuracy: 0.6389 - val_loss: 0.6636
Epoch 5/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6955 - loss: 0.5696 - val_accuracy: 0.6250 - val_loss: 0.6579
Epoch 6/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7289 - loss: 0.5718 - val_accuracy: 0.6528 - val_loss: 0.6527
Epoch 7/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7338 - loss: 0.5607 - val_accuracy: 0.6597 - val_loss: 0.6535
Epoch 8/20
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7506 - loss: 0.5308 - val_accuracy: 0.6528 - val_loss

**Reasoning**:
The model has been built and trained. The next logical step is to evaluate its performance on the test set using appropriate metrics, as stated in the overall task description. This will involve making predictions on the test data and calculating the accuracy.



In [9]:
from sklearn.metrics import accuracy_score

# === 7. Оценка ===
y_pred = (model.predict(X_test) > 0.5).astype(int)
print("Accuracy:", accuracy_score(y_test, y_pred))

print("Model evaluation complete. Accuracy score calculated.")

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
Accuracy: 0.6611111111111111
Model evaluation complete. Accuracy score calculated.
